# EDA · Real Food Classifier

Análisis exploratorio del dataset de Open Food Facts para predecir el **grado de procesamiento NOVA** de un producto (1: sin procesar · 2: ingredientes culinarios · 3: procesado · 4: ultraprocesado).

Este notebook cubre la **fase 1** del proyecto: limpieza común y análisis. El resultado es un CSV limpio en `data/processed/` del que partimos las tres para entrenar los modelos.

## Paso 1 · Descargar y filtrar

### 1.1 Columnas disponibles

El export completo de Open Food Facts pesa 1,4 GB comprimido. Antes de cargarlo, leemos solo la cabecera (`nrows=0`) para ver qué columnas tiene y elegir las que nos interesan.

In [1]:
import pandas as pd

RAW = "../data/raw/off_products.csv.gz"

cols = pd.read_csv(RAW, sep="\t", nrows=0).columns
print(f"Columnas totales: {len(cols)}")

interesantes = [c for c in cols if any(k in c for k in ["nova", "_100g", "pnns", "additives", "ingredients_text", "countries"])]
interesantes

Columnas totales: 211


['countries',
 'countries_tags',
 'countries_en',
 'ingredients_text',
 'additives_n',
 'additives',
 'additives_tags',
 'additives_en',
 'nova_group',
 'pnns_groups_1',
 'pnns_groups_2',
 'energy-kj_100g',
 'energy-kcal_100g',
 'energy_100g',
 'energy-from-fat_100g',
 'fat_100g',
 'saturated-fat_100g',
 'butyric-acid_100g',
 'caproic-acid_100g',
 'caprylic-acid_100g',
 'capric-acid_100g',
 'lauric-acid_100g',
 'myristic-acid_100g',
 'palmitic-acid_100g',
 'stearic-acid_100g',
 'arachidic-acid_100g',
 'behenic-acid_100g',
 'lignoceric-acid_100g',
 'cerotic-acid_100g',
 'montanic-acid_100g',
 'melissic-acid_100g',
 'unsaturated-fat_100g',
 'monounsaturated-fat_100g',
 'omega-9-fat_100g',
 'polyunsaturated-fat_100g',
 'omega-3-fat_100g',
 'omega-6-fat_100g',
 'alpha-linolenic-acid_100g',
 'eicosapentaenoic-acid_100g',
 'docosahexaenoic-acid_100g',
 'linoleic-acid_100g',
 'arachidonic-acid_100g',
 'gamma-linolenic-acid_100g',
 'dihomo-gamma-linolenic-acid_100g',
 'oleic-acid_100g',
 'elai

### 1.2 Carga por trozos y filtro a España

Para no llenar la memoria:
- leemos el fichero en trozos de 200.000 filas (`chunksize`);
- cargamos solo 23 de las 211 columnas (`usecols`): identificación, categoría, ingredientes (texto y normalizados), aditivos, NOVA y nutrientes principales por 100 g;
- nos quedamos solo con productos vendidos en España (`countries_tags` contiene `en:spain`).
- leemos `code` como texto (`dtype={"code": str}`). Si no, pandas lo interpreta como número y se pierden los ceros iniciales de los códigos de barras (`0011210115644` → `11210115644`). En la primera versión afectaba a 1.251 productos del CSV limpio.

**No cargamos** `nutriscore_score` ni `nutriscore_grade`: el objetivo es NOVA y el Nutri-Score no debe entrar al modelo.

El resultado se guarda en `data/interim/off_spain.csv.gz` para no repetir esta lectura, que tarda varios minutos.

In [2]:
USECOLS = [
    "code", "product_name", "brands",
    "countries_tags", "categories_tags", "pnns_groups_1", "pnns_groups_2",
    "ingredients_text", "additives_n", "additives_tags",
    "nova_group",
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g",
    "carbohydrates_100g", "sugars_100g", "fiber_100g",
    "proteins_100g", "salt_100g", "sodium_100g",
    "fruits-vegetables-legumes_100g", "ingredients_tags", "ingredients_analysis_tags",
]

faltan = [c for c in USECOLS if c not in cols]
print("Columnas que no existen:", faltan)

trozos = []
total = 0
for chunk in pd.read_csv(
        RAW, sep="\t", usecols=USECOLS,
        dtype={"code": str},
        chunksize=200_000,
        low_memory=False,
        on_bad_lines="skip",
):
    total += len(chunk)
    es = chunk[chunk["countries_tags"].str.contains("en:spain", na=False)]
    trozos.append(es)

df = pd.concat(trozos, ignore_index=True)
print(f"Filas totales: {total:,} -> España: {len(df):,}")

df.to_csv("../data/interim/off_spain.csv.gz", index=False)

Columnas que no existen: []
Filas totales: 4,906,687 -> España: 362,954


### 1.3 Embudo de filas y distribución de NOVA

Contamos cuántos productos sobreviven a cada filtro y cómo se reparten las clases.

In [3]:
print(f"Productos en España:        {len(df):,}")
print(f"Con NOVA:                   {df['nova_group'].notna().sum():,}")
print(f"Con NOVA e ingredientes:    {(df['nova_group'].notna() & df['ingredients_text'].notna()).sum():,}")

df["nova_group"].value_counts(dropna=False).sort_index()

Productos en España:        362,954
Con NOVA:                   54,403
Con NOVA e ingredientes:    46,561


nova_group
1.0      6996
2.0      8038
3.0     10390
4.0     28979
NaN    308551
Name: count, dtype: int64

**Resultados:**

| | Productos | % sobre el anterior |
|---|---|---|
| Export completo | 4.906.687 | |
| En España | 362.954 | 7,4 % |
| Con NOVA | 54.403 | 15 % |
| Con NOVA e ingredientes | 46.561 | 86 % |

| NOVA | Productos | % |
|---|---|---|
| 1 · sin procesar | 6.996 | 12,9 % |
| 2 · ingredientes culinarios | 8.038 | 14,8 % |
| 3 · procesados | 10.390 | 19,1 % |
| 4 · ultraprocesados | 28.979 | 53,3 % |

**Decisiones:**
- El 85 % de los productos españoles no tiene NOVA y no sirve para entrenar. Con ~54.000 productos etiquetados hay datos suficientes, así que **no ampliamos a otros países**.
- Desbalance moderado: la clase 4 es la mitad del dataset. **Mantenemos las 4 clases** y evaluaremos con **F1 macro** y recall por clase, no solo con accuracy.
- `nova_group` aparece como decimal (`1.0`) por los nulos. Se convertirá a entero al eliminarlos.

### 1.4 Productos con NOVA

Eliminamos los productos sin NOVA (no conocemos su clase, así que no sirven para entrenar) y pasamos `nova_group` a entero.

A partir de aquí cargamos el fichero intermedio, sin repetir la lectura del export completo.

In [4]:
df = pd.read_csv(
    "../data/interim/off_spain.csv.gz",
    low_memory=False,
    dtype={"code": str},
)

df = df[df["nova_group"].notna()].copy()

df["nova_group"] = df["nova_group"].astype(int)

print(df.shape)
df["nova_group"].value_counts().sort_index()

(54403, 23)


nova_group
1     6996
2     8038
3    10390
4    28979
Name: count, dtype: int64

## Paso 2 · Columnas que se eliminan

Al elegir `USECOLS` ya dejamos fuera las columnas con fuga de datos (Nutri-Score) y los metadatos (URLs, fechas, autor). Ahora revisamos las columnas cargadas: porcentaje de nulos y si aportan algo al modelo.

In [5]:
nulos = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
nulos

additives_tags                    47.6
fiber_100g                        39.6
fruits-vegetables-legumes_100g    17.0
ingredients_tags                  14.5
ingredients_text                  14.4
ingredients_analysis_tags         14.4
additives_n                       14.4
saturated-fat_100g                12.0
sodium_100g                       11.8
salt_100g                         11.8
sugars_100g                       11.7
proteins_100g                     11.2
fat_100g                          11.1
carbohydrates_100g                11.0
brands                            10.9
energy-kcal_100g                  10.4
categories_tags                    4.9
product_name                       2.4
code                               0.0
pnns_groups_2                      0.0
nova_group                         0.0
countries_tags                     0.0
pnns_groups_1                      0.0
dtype: float64

In [6]:
df.head()

,code,product_name,brands,categories_tags,countries_tags,ingredients_text,ingredients_tags,ingredients_analysis_tags,additives_n,additives_tags,...,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,fruits-vegetables-legumes_100g
0,0009800300037,gum cool watermelon,Tic Tac,"en:snacks,en:sweet-snacks,en:confectioneries,e...","en:france,en:spain,en:united-states","XYLITOL, GUM BASE, ISOMALT, GUM ARABIC, SORBIT...","en:e967,en:gum-base,en:e953,en:e414,en:e420,en...","en:palm-oil-free,en:vegan-status-unknown,en:ve...",12.0,"en:e129,en:e321,en:e334,en:e414,en:e420,en:e55...",...,333.0,0.0,0.0,67.0,0.0,NaN,0.0,0.000000,0.000000,0.000000
1,8422947520014,Seitán en lonchas,Soria Natural,"en:plant-based-foods-and-beverages,en:plant-ba...",en:spain,"_Seitán_ 83% (agua, gluten de _trigo_, harina ...","en:seitan,en:cereal,en:wheat,es:caldo-de-cocci...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,...,139.0,0.5,0.1,13.0,0.5,2.2,20.0,0.850000,0.340000,5.244531
2,00013029,Cheddar,McDonald's,"en:dairies,en:fermented-foods,en:fermented-mil...","en:denmark,en:france,en:norway,en:poland,en:sp...","_Cheddar_ (51%), eau, _fromage_ (9%), _lait_ é...","en:cheddar,en:dairy,en:cheese,en:water,en:skim...","en:may-contain-palm-oil,en:non-vegan,en:maybe-...",5.0,"en:e160a,en:e160ai,en:e160c,en:e322,en:e322i,e...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
3,0011210115644,Tabasco Rojo,McIlhenny Company,"en:condiments,en:sauces,en:hot-sauces,en:groce...","en:belgium,en:spain","Vinagre, chile rojo (19%), sal.","en:vinegar,en:red-chili-pepper,en:chili-pepper...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,...,16.0,0.7,0.2,1.6,0.0,1.3,1.0,1.800000,0.720000,19.000000
5,0012000171741,Diet Pepsi,Pepsi,"en:beverages-and-beverages-preparations,en:bev...","en:spain,en:united-states","Carbonated water, caramel color, aspartame, ph...","en:carbonated-water,en:water,en:e150a,en:e951,...","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",6.0,"en:e150a,en:e212,en:e330,en:e338,en:e950,en:e951",...,0.0,0.0,NaN,0.0,NaN,NaN,0.0,0.025381,0.010152,0.000000


### 2.1 Comprobaciones antes de decidir

- ¿Los nulos de `additives_tags` son productos sin aditivos?
- ¿Faltan los nutrientes todos a la vez?
- ¿`pnns_groups` esconde valores desconocidos como texto?
- ¿Hay alguna columna de ingredientes normalizada?

In [7]:
# 1. Nulos de additives_tags
sin_tags = df["additives_tags"].isna()
print("additives_tags nulos, según additives_n:")
print(df.loc[sin_tags, "additives_n"].value_counts(dropna=False).head())

# 2. ¿Cuántos productos no tienen NINGÚN nutriente principal?
NUTRIENTES = ["energy-kcal_100g", "fat_100g", "saturated-fat_100g",
              "carbohydrates_100g", "sugars_100g", "proteins_100g", "salt_100g"]
print("\nSin ningún nutriente:", df[NUTRIENTES].isna().all(axis=1).sum())
print("Con algún nutriente pero no todos:", (df[NUTRIENTES].isna().any(axis=1) & df[NUTRIENTES].notna().any(axis=1)).sum())

# 3. Valores de pnns: "unknown" o similares
print("\npnns_groups_1:")
print(df["pnns_groups_1"].value_counts())

# 4. Columnas de ingredientes en el export original (cols es de la celda 1)
print("\nColumnas de ingredientes:", [c for c in cols if "ingredient" in c])

additives_tags nulos, según additives_n:
additives_n
0.0    18075
NaN     7842
Name: count, dtype: int64

Sin ningún nutriente: 5097
Con algún nutriente pero no todos: 2645

pnns_groups_1:
pnns_groups_1
Sugary snacks              11031
unknown                     8423
Fat and sauces              7193
Cereals and potatoes        6181
Beverages                   5221
Milk and dairy products     4865
Fish Meat Eggs              3643
Fruits and vegetables       3122
Salty snacks                2180
Composite foods             1832
Alcoholic beverages          549
Baby foods                   162
Name: count, dtype: int64

Columnas de ingredientes: ['ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags', 'image_ingredients_url', 'image_ingredients_small_url']


### 2.2 Decisiones

- `additives_tags`: 18.075 nulos son productos **sin aditivos** (`additives_n = 0`), no datos perdidos. Se rellenan con lista vacía en el paso 5.
- 5.099 productos no tienen **ningún** nutriente: se eliminan en el paso 5. Otros 2.645 tienen nutrientes parciales: se imputan.
- `pnns_groups_1` tiene 8.423 valores `"unknown"`. Se conservan como categoría propia.
- Se ha añadido `ingredients_tags` (ingredientes normalizados en inglés) e `ingredients_analysis_tags` para crear variables en el paso 7.

**Columnas eliminadas:**
- `countries_tags`: tras filtrar por España no aporta información.
- `categories_tags`: lista de categorías muy larga y variable; `pnns_groups_1` y `pnns_groups_2` ya resumen la categoría del producto.

**Columnas que no entran al modelo pero se conservan:** `code`, `product_name` y `brands`, para detectar duplicados (paso 6) y para la app.

In [8]:
df = df.drop(columns=["countries_tags", "categories_tags"])

# Identificadores
ID_COLS = ["code", "product_name", "brands"]

print(df.shape)
df.columns.tolist()

(54403, 21)


['code',
 'product_name',
 'brands',
 'ingredients_text',
 'ingredients_tags',
 'ingredients_analysis_tags',
 'additives_n',
 'additives_tags',
 'nova_group',
 'pnns_groups_1',
 'pnns_groups_2',
 'energy-kcal_100g',
 'fat_100g',
 'saturated-fat_100g',
 'carbohydrates_100g',
 'sugars_100g',
 'fiber_100g',
 'proteins_100g',
 'salt_100g',
 'sodium_100g',
 'fruits-vegetables-legumes_100g']

## Paso 3 · Valores sucios y unidades

Comprobamos:
- que las columnas numéricas se han leído como números;
- que `salt_100g` y `sodium_100g` dicen lo mismo (sal ≈ sodio × 2,5);
- qué valores tiene `pnns_groups_2`;
- el formato de las columnas de ingredientes normalizados.

In [9]:
df.dtypes

code                                  str
product_name                          str
brands                                str
ingredients_text                      str
ingredients_tags                      str
ingredients_analysis_tags             str
additives_n                       float64
additives_tags                        str
nova_group                          int64
pnns_groups_1                         str
pnns_groups_2                         str
energy-kcal_100g                  float64
fat_100g                          float64
saturated-fat_100g                float64
carbohydrates_100g                float64
sugars_100g                       float64
fiber_100g                        float64
proteins_100g                     float64
salt_100g                         float64
sodium_100g                       float64
fruits-vegetables-legumes_100g    float64
dtype: object

In [10]:
# Solo filas con las dos columnas y sodio > 0 (para no dividir entre cero)
mask = df["salt_100g"].notna() & df["sodium_100g"].notna() & (df["sodium_100g"] > 0)
ratio = df.loc[mask, "salt_100g"] / df.loc[mask, "sodium_100g"]

# Si es la misma información, el ratio será ~2,5 en casi todas las filas
print(ratio.describe())
print("\n% de filas con ratio entre 2,4 y 2,6:", round(ratio.between(2.4, 2.6).mean() * 100, 1))

# ¿Hay productos con sodio pero sin sal? Esos se podrían recuperar
print("Sodio sin sal:", (df["salt_100g"].isna() & df["sodium_100g"].notna()).sum())
print("Sal sin sodio:", (df["salt_100g"].notna() & df["sodium_100g"].isna()).sum())

count    3.974100e+04
mean     1.524583e+17
std      3.039279e+19
min      0.000000e+00
25%      2.500000e+00
50%      2.500000e+00
75%      2.500000e+00
max      6.058847e+21
dtype: float64

% de filas con ratio entre 2,4 y 2,6: 97.4
Sodio sin sal: 0
Sal sin sodio: 0


In [11]:
# ¿Esconde también "unknown"? ¿Hay grupos muy pequeños?
df["pnns_groups_2"].value_counts()

pnns_groups_2
unknown                             8423
Sweets                              5903
Fats                                4571
Biscuits and cakes                  3418
Dressings and sauces                2622
Cereals                             2061
Milk and yogurt                     1900
Vegetables                          1862
Cheese                              1781
Bread                               1697
Fish and seafood                    1549
Chocolate products                  1477
One-dish meals                      1255
Breakfast cereals                   1140
Artificially sweetened beverages    1139
Processed meat                      1094
Waters and flavored waters           929
Meat                                 895
Appetizers                           872
Plant-based milk substitutes         825
Teas and herbal teas and coffees     818
Sweetened beverages                  817
Legumes                              711
Ice cream                            673
Sa

In [12]:
# Listas
for col in ["ingredients_tags", "ingredients_analysis_tags", "additives_tags"]:
    print(col)
    print(df[col].dropna().iloc[0][:300], "\n")

ingredients_tags
en:e967,en:gum-base,en:e953,en:e414,en:e420,en:natural-and-artificial-flavouring,en:artificial-flavouring,en:natural-flavouring,en:flavouring,en:e553b,en:e553,en:LESS THAN 2% OF,en:e950,en:e334,en:e955,en:e903,en:e129,en:e321,en:CITRIC ACID MAGNESIUM STEARATE,en:TO MAINTAIN FREHSNESS 

ingredients_analysis_tags
en:palm-oil-free,en:vegan-status-unknown,en:vegetarian-status-unknown 

additives_tags
en:e129,en:e321,en:e334,en:e414,en:e420,en:e553,en:e553b,en:e572,en:e903,en:e950,en:e953,en:e955,en:e967 



In [13]:
# Un tag "limpio" tiene prefijo de idioma y palabras en minúscula con guiones: en:palm-oil
patron_limpio = r"^[a-z]{2}:[a-z0-9-]+$"

tags = df["ingredients_tags"].dropna().str.split(",").explode()
print(f"Tags totales: {len(tags):,}")
print(f"% limpios:    {tags.str.match(patron_limpio).mean() * 100:.1f}")

# ¿Qué idiomas aparecen? (si hay muchos es:/fr:, son ingredientes sin traducir)
print("\nPrefijos:")
print(tags.str[:3].value_counts().head(8))

Tags totales: 1,052,091
% limpios:    95.0

Prefijos:
ingredients_tags
en:    946124
es:     68877
fr:     12388
ar:      8991
pt:      4177
de:      2349
ca:      2340
it:      1540
Name: count, dtype: int64


In [14]:
df = df.drop(columns=["sodium_100g"])
print(df.shape)

(54403, 20)


### 3.1 Decisiones

- **Tipos:** correctos. No hay conversiones pendientes.
- **Sal y sodio:** en el 97,4 % de las filas sal = sodio × 2,5 y sus nulos coinciden. Se elimina `sodium_100g`. Los ratios absurdos (2,6 %) son errores de tecleo y se tratan en el paso 4.
- **`pnns_groups_2`:** 42 subcategorías, con los mismos 8.423 `unknown` que `pnns_groups_1`. Se conserva para imputar nulos (paso 5); qué nivel de categoría entra al modelo se decide en el paso 10.
- **Listas de ingredientes y aditivos:** texto separado por comas con prefijo de idioma. En `ingredients_tags`, ~90 % de los tags están reconocidos en inglés (`en:`), ~6,5 % quedan en español sin traducir (`es:`) y ~5 % son texto sin reconocer. Además, incluye los ingredientes "padre" (por ejemplo `flavouring` junto a `natural-flavouring`), así que no sirve para contar ingredientes directamente. `additives_tags` e `ingredients_analysis_tags` están limpias. Se trabajará en el paso 7.

## Paso 4 · Valores atípicos

Distinguimos dos tipos:
- **Imposibles** (errores de datos): más de 100 g de un nutriente por 100 g, negativos, azúcares mayores que hidratos, saturadas mayores que grasa total, energía imposible. Se pasan a nulo y se imputan en el paso 5.
- **Extremos reales** (aceite, sal, azúcar): se conservan.

No usamos la regla IQR para borrar filas: un aceite con 100 g de grasa es un extremo pero es real.

In [15]:
NUM = ["energy-kcal_100g", "fat_100g", "saturated-fat_100g", "carbohydrates_100g",
       "sugars_100g", "fiber_100g", "proteins_100g", "salt_100g",
       "fruits-vegetables-legumes_100g", "additives_n"]

# min, max y percentiles
df[NUM].describe(percentiles=[0.01, 0.5, 0.99]).T.round(1)

,count,mean,std,min,1%,50%,99%,max
energy-kcal_100g,48719.0,309.4,342.1,0.0,0.0,279.0,900.0,39006.0
fat_100g,48342.0,19.0,29.8,-1.4,0.0,7.6,100.0,2800.0
saturated-fat_100g,47848.0,5.6,9.4,-4.0,0.0,1.8,34.7,605.0
carbohydrates_100g,48412.0,26.4,28.8,0.0,0.0,12.0,95.0,1160.0
sugars_100g,48017.0,11.8,18.9,0.0,0.0,3.1,80.9,420.0
fiber_100g,32843.0,3.5,13.0,-0.2,0.0,1.7,27.0,2020.0
proteins_100g,48331.0,7.4,12.2,0.0,0.0,4.9,36.9,1600.0
salt_100g,47990.0,1.5,37.3,0.0,0.0,0.2,11.5,7017.0
fruits-vegetables-legumes_100g,45144.0,16.0,33.0,-2.2,0.0,0.0,100.0,846.0
additives_n,46561.0,2.1,3.1,0.0,0.0,1.0,12.0,319.0


In [16]:
gramos = ["fat_100g", "saturated-fat_100g", "carbohydrates_100g", "sugars_100g",
          "fiber_100g", "proteins_100g", "salt_100g", "fruits-vegetables-legumes_100g"]

reglas = {}

# 1. Gramos por 100 g fuera de 0-100
for c in gramos:
    reglas[f"{c} fuera de 0-100"] = ((df[c] < 0) | (df[c] > 100)).sum()

# 2. Energía: la grasa pura tiene ~900 kcal/100 g, nada puede pasar de ahí
reglas["energía < 0 o > 900"] = ((df["energy-kcal_100g"] < 0) | (df["energy-kcal_100g"] > 900)).sum()

# 3. Una parte no puede ser mayor que el total (+0,5 g de margen por redondeos)
reglas["azúcares > hidratos"] = (df["sugars_100g"] > df["carbohydrates_100g"] + 0.5).sum()
reglas["saturadas > grasa"] = (df["saturated-fat_100g"] > df["fat_100g"] + 0.5).sum()

# 4. Los macronutrientes juntos no pueden pasar de 100 g (+1 g de margen)
suma = df[["fat_100g", "carbohydrates_100g", "proteins_100g", "salt_100g"]].sum(axis=1, min_count=1)
reglas["grasa+hidratos+proteína+sal > 101"] = (suma > 101).sum()

# 5. Aditivos: comprobamos el máximo para ver si hay valores raros
reglas["additives_n > 30"] = (df["additives_n"] > 30).sum()

pd.Series(reglas, name="filas").to_frame()

,filas
fat_100g fuera de 0-100,8
saturated-fat_100g fuera de 0-100,4
carbohydrates_100g fuera de 0-100,8
sugars_100g fuera de 0-100,5
fiber_100g fuera de 0-100,5
proteins_100g fuera de 0-100,5
salt_100g fuera de 0-100,20
fruits-vegetables-legumes_100g fuera de 0-100,429
energía < 0 o > 900,78
azúcares > hidratos,66


In [17]:
# Energía calculada con los factores estándar: 9 kcal/g grasa, 4 hidratos, 4 proteína, 2 fibra
estimada = (9 * df["fat_100g"] + 4 * df["carbohydrates_100g"]
            + 4 * df["proteins_100g"] + 2 * df["fiber_100g"].fillna(0))

diff = (df["energy-kcal_100g"] - estimada).abs()

# Diferencia en kcal: los percentiles nos dicen cuánta desviación es "normal"
print(diff.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))
print("\nDiferencia > 100 kcal:", (diff > 100).sum())

count    48199.0
mean        11.4
std        191.7
min          0.0
50%          0.9
90%         13.9
95%         34.0
99%        156.9
max      34328.0
dtype: float64

Diferencia > 100 kcal: 874


In [18]:
import numpy as np

antes = len(df)
macros_g = ["fat_100g", "saturated-fat_100g", "carbohydrates_100g", "sugars_100g",
            "fiber_100g", "proteins_100g", "salt_100g"]

# 1. Gramos fuera de 0-100 → nulo
for c in macros_g:
    df.loc[(df[c] < 0) | (df[c] > 100), c] = np.nan

# 2. Fruta/verdura: estimación de OFF, se recorta al rango 0-100
df["fruits-vegetables-legumes_100g"] = df["fruits-vegetables-legumes_100g"].clip(0, 100)

# 3. Energía imposible → nulo
df.loc[(df["energy-kcal_100g"] < 0) | (df["energy-kcal_100g"] > 900), "energy-kcal_100g"] = np.nan

# 4. Una parte mayor que su total: no sabemos cuál falla → los dos a nulo
m = df["sugars_100g"] > df["carbohydrates_100g"] + 0.5
df.loc[m, ["sugars_100g", "carbohydrates_100g"]] = np.nan
m = df["saturated-fat_100g"] > df["fat_100g"] + 0.5
df.loc[m, ["saturated-fat_100g", "fat_100g"]] = np.nan

# 5. Macros que suman más de 101 g: etiqueta incoherente → se elimina el producto
suma = df[["fat_100g", "carbohydrates_100g", "proteins_100g", "salt_100g"]].sum(axis=1, min_count=1)
df = df[~(suma > 101)].copy()

# 6. Energía que no cuadra con los macros (salvo alcohol) → nulo; se recalcula en el paso 5
estimada = (9 * df["fat_100g"] + 4 * df["carbohydrates_100g"]
            + 4 * df["proteins_100g"] + 2 * df["fiber_100g"].fillna(0))
m = ((df["energy-kcal_100g"] - estimada).abs() > 100) & (df["pnns_groups_1"] != "Alcoholic beverages")
df.loc[m, "energy-kcal_100g"] = np.nan

# 7. Número de aditivos absurdo → nulo
df.loc[df["additives_n"] > 30, "additives_n"] = np.nan

print(f"Filas: {antes:,} -> {len(df):,}")
df[NUM].describe().T[["count", "min", "max"]]

Filas: 54,403 -> 54,288


,count,min,max
energy-kcal_100g,47758.0,0.0,900.0
fat_100g,48193.0,0.0,100.0
saturated-fat_100g,47705.0,0.0,100.0
carbohydrates_100g,48228.0,0.0,100.0
sugars_100g,47837.0,0.0,100.0
fiber_100g,32756.0,0.0,87.0
proteins_100g,48212.0,0.0,100.0
salt_100g,47860.0,0.0,100.0
fruits-vegetables-legumes_100g,45049.0,0.0,100.0
additives_n,46460.0,0.0,27.0


### 4.1 Decisiones

Hay muy pocos valores imposibles: decenas de filas por regla sobre 54.403.

- **Nutrientes fuera de 0-100 g, energía fuera de 0-900 kcal y más de 30 aditivos:** errores de tecleo → nulo.
- **Fruta, verdura y legumbres:** es una estimación de Open Food Facts a partir de los ingredientes (puede pasar de 100 en productos concentrados) → se recorta al rango 0-100.
- **Azúcares > hidratos y saturadas > grasa:** no se sabe cuál de los dos valores falla → ambos a nulo.
- **Macros que suman más de 101 g (115 productos, una vez anulados los valores imposibles):** etiqueta incoherente en varios valores a la vez → **se elimina el producto**. Es la única excepción a no borrar filas.
- **Energía incoherente con los macros (870 productos, sin contar bebidas alcohólicas):** → nulo. En el paso 5 se recalcula con la fórmula 9·grasa + 4·hidratos + 4·proteína + 2·fibra.
- **Extremos reales** (aceites, sal, azúcar): se conservan.

Filas: 54.403 → 54.288

## Paso 5 · Valores nulos

En la fase 1 solo hacemos arreglos **deterministas**, que no dependen de otros productos. La imputación con medianas se hace en la fase 2, dentro del pipeline y calculada solo con train, para no filtrar información del test.

### 5.1 Comprobaciones

In [19]:
NUTRIENTES = ["energy-kcal_100g", "fat_100g", "saturated-fat_100g",
              "carbohydrates_100g", "sugars_100g", "proteins_100g", "salt_100g"]

sin_nutrientes = df[NUTRIENTES].isna().all(axis=1)
sin_ingredientes = df["ingredients_text"].isna()

print(f"Sin ningún nutriente: {sin_nutrientes.sum():,}")
print(f"Sin ingredientes:     {sin_ingredientes.sum():,}")
print(f"Sin ninguna de las dos cosas: {(sin_nutrientes & sin_ingredientes).sum():,}")

# ¿Cómo se reparten por clase los productos sin ingredientes, comparado con el total?
comparacion = pd.DataFrame({
    "todos (%)": df["nova_group"].value_counts(normalize=True).sort_index().mul(100).round(1),
    "sin ingredientes (%)": df.loc[sin_ingredientes, "nova_group"].value_counts(normalize=True).sort_index().mul(100).round(1),
    "sin ingredientes (n)": df.loc[sin_ingredientes, "nova_group"].value_counts().sort_index(),
})
comparacion

Sin ningún nutriente: 5,099
Sin ingredientes:     7,825
Sin ninguna de las dos cosas: 2,665


,todos (%),sin ingredientes (%),sin ingredientes (n)
nova_group,,,
1,12.9,9.1,710.0
2,14.8,90.9,7115.0
3,19.1,NaN,NaN
4,53.3,NaN,NaN


In [20]:
# ¿Qué tipo de productos son los que no tienen ingredientes?
df.loc[sin_ingredientes, "pnns_groups_2"].value_counts().head(10)

pnns_groups_2
Fats                          3882
Sweets                        2241
Waters and flavored waters     705
Dressings and sauces           546
unknown                        374
Cereals                         44
Fruits                          12
Salty and fatty products         6
Baby foods                       2
Sweetened beverages              2
Name: count, dtype: int64

In [21]:
antes = len(df)

# 1. Sin ningún nutriente: no pueden pasar por el formulario de la app → se eliminan
sin_nutrientes = df[NUTRIENTES].isna().all(axis=1)
df = df[~sin_nutrientes].copy()

# 2. Marca: producto sin lista de ingredientes (en la UE, normalmente un solo ingrediente)
df["sin_lista_ingredientes"] = df["ingredients_text"].isna().astype(int)

# 3. Aditivos: un producto sin lista no tiene aditivos → 0
df.loc[df["sin_lista_ingredientes"] == 1, "additives_n"] = 0
#    y quien tiene 0 aditivos tiene lista de aditivos vacía, no un dato perdido
df.loc[df["additives_n"] == 0, "additives_tags"] = ""

# 4. Energía: si falta y tenemos los macros, se recalcula con la fórmula (salvo alcohol)
falta_energia = (df["energy-kcal_100g"].isna()
                 & df[["fat_100g", "carbohydrates_100g", "proteins_100g"]].notna().all(axis=1)
                 & (df["pnns_groups_1"] != "Alcoholic beverages"))
df.loc[falta_energia, "energy-kcal_100g"] = (
        9 * df.loc[falta_energia, "fat_100g"]
        + 4 * df.loc[falta_energia, "carbohydrates_100g"]
        + 4 * df.loc[falta_energia, "proteins_100g"]
        + 2 * df.loc[falta_energia, "fiber_100g"].fillna(0)
)

print(f"Filas: {antes:,} -> {len(df):,}")
print(f"Energía recalculada: {falta_energia.sum():,}")

Filas: 54,288 -> 49,189
Energía recalculada: 840


In [22]:
# Nulos que quedan: se imputarán en la fase 2, dentro del pipeline
print(df.isna().mean().mul(100).round(1).sort_values(ascending=False).head(12))

# ¿Sigue viva la clase 2 tras eliminar los productos sin nutrientes?
df["nova_group"].value_counts().sort_index()

fiber_100g                        33.4
fruits-vegetables-legumes_100g    11.4
ingredients_tags                  10.6
ingredients_text                  10.5
ingredients_analysis_tags         10.4
brands                             8.4
saturated-fat_100g                 3.0
sugars_100g                        2.7
salt_100g                          2.7
proteins_100g                      2.0
fat_100g                           2.0
carbohydrates_100g                 2.0
dtype: float64


nova_group
1     6344
2     5647
3    10046
4    27152
Name: count, dtype: int64

### 5.2 Decisiones

- **Productos sin ningún nutriente (5.099):** se eliminan, porque sin tabla nutricional no pueden usarse en la app. Muchos no son errores: en la UE hay productos exentos de tabla (un solo ingrediente sin procesar, agua, vinagre, sal, especias). Por eso afecta sobre todo a la clase 2, que pierde un 30 % (8.038 → 5.647), aunque conserva productos suficientes. **Limitación conocida:** el modelo no está pensado para productos sin tabla nutricional.
- **Productos sin lista de ingredientes (7.825):** el 91 % son NOVA 2 (aceites, azúcar, miel, aguas, vinagres). Son productos de **un solo ingrediente**, que en la UE no necesitan lista. Que falte la lista es información, no un dato perdido:
  - se **conservan** (eliminarlos dejaría la clase 2 con ~900 productos);
  - se crea la variable `sin_lista_ingredientes` (0/1). En la app será una casilla del formulario;
  - su `additives_n` es 0.
- **`additives_tags`:** vacía cuando `additives_n = 0`.
- **Energía:** cuando falta y hay grasa, hidratos y proteína, se recalcula con 9·grasa + 4·hidratos + 4·proteína + 2·fibra (salvo bebidas alcohólicas).
- **Resto de nulos:** se quedan en el CSV. Se imputan en la fase 2, dentro del pipeline y solo con train.

Resultado: 54.288 → **49.189 productos**. Energía recalculada en 840. Los nulos que quedan (fibra 33 %, fruta y verdura 11 %, resto de nutrientes ≤ 3 %) se imputan en la fase 2.

| NOVA | Productos | % |
|---|---|---|
| 1 | 6.344 | 12,9 % |
| 2 | 5.647 | 11,5 % |
| 3 | 10.046 | 20,4 % |
| 4 | 27.152 | 55,2 % |

## Paso 6 · Duplicados y variantes

Buscamos:
- códigos de barras repetidos (duplicados reales);
- el mismo producto con distinto código (formatos, packs, ediciones): mismo nombre, marca y nutrientes. Si una copia cae en train y otra en test, la métrica sale inflada;
- productos iguales con NOVA distinto (etiquetas contradictorias).

In [23]:
# 1. Códigos de barras repetidos
print("Códigos repetidos:", df["code"].duplicated().sum())

# 2. Mismo nombre + marca, normalizados (minúsculas, sin espacios sobrantes)
#    Solo productos que tienen las dos cosas: si no, todos los nulos contarían como "iguales"
nombre = df["product_name"].str.lower().str.strip()
marca = df["brands"].str.lower().str.strip()
con_ambos = nombre.notna() & marca.notna()

sub = df[con_ambos].assign(clave=nombre[con_ambos] + " | " + marca[con_ambos])
print("Productos con nombre y marca:", len(sub))
print("Comparten nombre + marca:", sub["clave"].duplicated(keep=False).sum())

# 3. Además, los mismos nutrientes → el mismo producto en otro formato
copias = sub.duplicated(subset=["clave"] + NUTRIENTES, keep="first")
print("Copias con mismo nombre, marca y nutrientes:", copias.sum())

# 4. Mismo nombre + marca con NOVA distinto
nova_por_clave = sub.groupby("clave")["nova_group"].nunique()
print("Nombre + marca con NOVA distinto:", (nova_por_clave > 1).sum())

Códigos repetidos: 4
Productos con nombre y marca: 44360
Comparten nombre + marca: 4850
Copias con mismo nombre, marca y nutrientes: 1039
Nombre + marca con NOVA distinto: 211


In [24]:
# Ejemplos de copias, para ver qué pinta tienen
(sub[sub.duplicated(subset=["clave"] + NUTRIENTES, keep=False)]
 .sort_values("clave")
 [["code", "product_name", "brands", "nova_group", "energy-kcal_100g", "fat_100g", "sugars_100g"]]
 .head(12))

,code,product_name,brands,nova_group,energy-kcal_100g,fat_100g,sugars_100g
19043,8413041318010,"0,0",Falke,1,19.0,0.0,1.9
21984,8413041328019,"0,0",Falke,1,19.0,0.0,1.9
74107,8413041318003,"0,0",Falke,1,19.0,0.0,1.9
16081,8480000511188,4 quesos,Hacendado,4,352.0,28.0,0.0
144468,17118880,4 quesos,Hacendado,4,352.0,28.0,0.0
57162,8435037801641,Aceite de coco virgen extra,Sol Natural,2,900.0,100.0,0.0
101823,8435037801658,Aceite de coco virgen extra,Sol Natural,2,900.0,100.0,0.0
13735,8432580271013,aceite de coco,la masía,2,900.0,100.0,0.0
18400,8432580271037,Aceite de coco,La Masía,2,900.0,100.0,0.0
149254,8425402529515,Aceite de coco,naturitas,2,900.0,100.0,0.0


In [25]:
antes = len(df)

# 1. Códigos de barras repetidos → nos quedamos con el primero
df = df.drop_duplicates(subset="code", keep="first").copy()

# 2. Grupo de producto: nombre + marca normalizados
#    Si falta alguno de los dos, el grupo es el propio código (producto único)
nombre = df["product_name"].str.lower().str.strip()
marca = df["brands"].str.lower().str.strip()
df["grupo_producto"] = (nombre + " | " + marca).fillna(df["code"])

# 3. Copias exactas (mismo grupo y mismos nutrientes) con NOVA contradictorio → fuera todas
claves_copia = ["grupo_producto"] + NUTRIENTES
n_nova = df.groupby(claves_copia, dropna=False)["nova_group"].transform("nunique")
contradictorias = n_nova > 1
print(f"Copias con NOVA contradictorio (eliminadas): {contradictorias.sum():,}")
df = df[~contradictorias].copy()

# 4. Resto de copias exactas → nos quedamos con una
n_antes_copias = len(df)
df = df.drop_duplicates(subset=claves_copia, keep="first").copy()
print(f"Copias exactas eliminadas: {n_antes_copias - len(df):,}")

print(f"\nFilas: {antes:,} -> {len(df):,}")
print(f"Grupos de producto: {df['grupo_producto'].nunique():,}")
print(f"Grupos con más de una versión: {(df['grupo_producto'].value_counts() > 1).sum():,}")
df["nova_group"].value_counts().sort_index()

Copias con NOVA contradictorio (eliminadas): 70
Copias exactas eliminadas: 996

Filas: 49,189 -> 48,119
Grupos de producto: 46,315
Grupos con más de una versión: 1,439


nova_group
1     6190
2     5385
3     9831
4    26713
Name: count, dtype: int64

### 6.1 Decisiones

- **Códigos de barras repetidos (4):** se eliminan.
- **Copias del mismo producto** (mismo nombre, marca y nutrientes, distinto código: formatos y packs):
  - si tienen **NOVA contradictorio** (70 productos): se eliminan todas, porque no sabemos cuál es la etiqueta correcta;
  - si no: se conserva una sola copia (996 eliminadas).
- **Mismo nombre y marca con nutrientes distintos:** se conservan, pueden ser reformulaciones.
- **Columna `grupo_producto`** (nombre + marca, o el código si faltan): el split fijo del paso 8 agrupa por `grupo_producto`; la CV de la fase 2 usa `StratifiedGroupKFold` con estos grupos, para que las versiones de un mismo producto no queden repartidas entre train y test. Esta regla es para las tres.

Resultado: 49.189 → **48.119 productos** en 46.315 grupos.

| NOVA | Productos | % |
|---|---|---|
| 1 | 6.190 | 12,9 % |
| 2 | 5.385 | 11,2 % |
| 3 | 9.831 | 20,4 % |
| 4 | 26.713 | 55,5 % |

Las proporciones apenas cambian respecto al paso 5: los duplicados se repartían por igual entre clases.

## Paso 7 · Variables a partir de ingredientes y aditivos

NOVA clasifica por **grado de procesamiento**, no por composición nutricional. Los ultraprocesados se reconocen por ingredientes que no se usan en una cocina doméstica (aromas, edulcorantes, jarabes, almidones modificados, grasas hidrogenadas, proteínas aisladas) y por ciertos aditivos.

Creamos variables sí/no a partir de `ingredients_tags` y `additives_tags` (normalizados en inglés) y el número de ingredientes a partir del texto de la etiqueta.

También marcamos los **ingredientes culinarios añadidos** (sal, azúcar, aceite) y los conservantes. NOVA 3 se define como un alimento natural al que se añaden ingredientes de NOVA 2, así que estos marcadores deberían ayudar a separar 1 y 3, donde los marcadores de ultraprocesado no aparecen.

**Aviso:** Open Food Facts asigna NOVA en parte con reglas basadas en estos mismos marcadores. Es esperable que estas variables sean muy predictivas.

In [26]:
# Listas como texto; vacío si no hay (producto sin lista → no tiene ningún marcador)
ing = df["ingredients_tags"].fillna("")
adi = df["additives_tags"].fillna("")
ana = df["ingredients_analysis_tags"].fillna("")

def tiene(texto, patron):
    """1 si el texto contiene el patrón (expresión regular), 0 si no."""
    return texto.str.contains(patron, regex=True).astype(int)

# Ingredientes marcadores de ultraprocesado
df["tiene_aromas"] = tiene(ing, r"en:flavouring")
df["tiene_edulcorantes"] = (tiene(ing, r"en:sweetener") | tiene(adi, r"en:e9[56]\d")).astype(int)
df["tiene_jarabes"] = tiene(ing, r"syrup|en:maltodextrin|en:dextrose")
df["tiene_almidon_modificado"] = (tiene(ing, r"modified-starch") | tiene(adi, r"en:e14\d\d")).astype(int)
df["tiene_grasa_hidrogenada"] = tiene(ing, r"(?<!non-)hydrogenated")    # (?<!non-): excluye "non-hydrogenated" (aceite NO hidrogenado)
df["tiene_proteina_aislada"] = tiene(ing, r"protein-isolate|whey|caseinate|hydroly[sz]ed")

# Familias de aditivos (por su número E)
df["tiene_colorantes"] = tiene(adi, r"en:e1\d\d(?!\d)")
df["tiene_emulsionantes"] = tiene(adi, r"en:e4\d\d")      # emulsionantes, espesantes, estabilizantes
df["tiene_potenciadores"] = tiene(adi, r"en:e6[23]\d")    # potenciadores del sabor (glutamato…)

# Aceite de palma: solo el tag exacto en:palm-oil (no "palm-oil-free" ni "unknown")
df["tiene_aceite_palma"] = tiene(ana, r"(?:^|,)en:palm-oil(?:,|$)")

# Conservantes: E200-E252 (sorbatos, benzoatos, sulfitos, nitritos) y E280-E285 (propionatos).
# Sin E260-E299 restantes: son acidulantes (acético, láctico, málico, CO2)
df["tiene_conservantes"] = tiene(adi, r"en:e2(?:[0-4]\d|5[0-2]|8[0-5])(?!\d)")

# Ingredientes culinarios añadidos: alimento natural + sal/azúcar/aceite = NOVA 3 (procesado)
# Tag exacto entre comas; ingredients_tags incluye los ingredientes "padre" (paso 3),
# así que una sal o un aceite concretos también llevan el tag genérico
df["tiene_sal"] = tiene(ing, r"(?:^|,)en:salt(?:,|$)")
df["tiene_azucar"] = tiene(ing, r"(?:^|,)en:sugar(?:,|$)")
df["tiene_aceite"] = tiene(ing, r"(?:^|,)en:(?:vegetable-oil|oil|[a-z-]*-oil)(?:,|$)")

FLAGS = [c for c in df.columns if c.startswith("tiene_")]
print(FLAGS)

['tiene_aromas', 'tiene_edulcorantes', 'tiene_jarabes', 'tiene_almidon_modificado', 'tiene_grasa_hidrogenada', 'tiene_proteina_aislada', 'tiene_colorantes', 'tiene_emulsionantes', 'tiene_potenciadores', 'tiene_aceite_palma', 'tiene_conservantes', 'tiene_sal', 'tiene_azucar', 'tiene_aceite']


In [27]:
import re

def contar_ingredientes(texto):
    """Cuenta los ingredientes principales: separa por comas, sin contar los de dentro de paréntesis."""
    if pd.isna(texto):
        return np.nan
    # Quitar paréntesis y corchetes (sub-ingredientes, porcentajes), también anidados
    anterior = None
    while anterior != texto:
        anterior = texto
        texto = re.sub(r"\([^()]*\)|\[[^\[\]]*\]", "", texto)
    partes = [p for p in re.split(r"[,;]", texto) if p.strip()]
    return len(partes)

df["n_ingredientes"] = df["ingredients_text"].apply(contar_ingredientes)
# Sin lista = un solo ingrediente (lo decidimos en el paso 5)
df.loc[df["sin_lista_ingredientes"] == 1, "n_ingredientes"] = 1

df["n_ingredientes"].describe()

count    48119.000000
mean         7.684906
std          7.865497
min          0.000000
25%          2.000000
50%          6.000000
75%         11.000000
max        279.000000
Name: n_ingredientes, dtype: float64

In [28]:
# Media de cada variable por clase. En las sí/no, la media es el % de productos que la tienen
(df.groupby("nova_group")[FLAGS + ["n_ingredientes", "additives_n"]]
 .mean()
 .round(2)
 .T)

nova_group,1,2,3,4
tiene_aromas,0.00,0.00,0.00,0.61
tiene_edulcorantes,0.00,0.00,0.00,0.13
tiene_jarabes,0.01,0.00,0.01,0.34
tiene_almidon_modificado,0.00,0.00,0.00,0.13
tiene_grasa_hidrogenada,0.00,0.00,0.00,0.01
tiene_proteina_aislada,0.00,0.00,0.00,0.11
tiene_colorantes,0.00,0.00,0.00,0.21
tiene_emulsionantes,0.00,0.00,0.00,0.47
tiene_potenciadores,0.00,0.00,0.00,0.05
tiene_aceite_palma,0.00,0.00,0.02,0.11


In [29]:
print("0 ingredientes:", (df["n_ingredientes"] == 0).sum())
print("Más de 50:", (df["n_ingredientes"] > 50).sum())

# Ejemplos de los extremos, para ver qué pasa con el texto
df.loc[df["n_ingredientes"] > 50, ["product_name", "n_ingredientes", "ingredients_text"]].head(3)

0 ingredientes: 2
Más de 50: 162


,product_name,n_ingredientes,ingredients_text
568,Formula 1 sostitutivo del pasto herbalife min...,60.0,"soy protein isolate (42%), fructose, oat fibre..."
650,Assorted fruit yogurts,77.0,"kiselosti: natrijevi citratei, bojilo: karmin,..."
1024,"Ecorino mini, passion fruit and peach & strawb...",56.0,wafelhoorntjes gedecoreerd met 9% dandheiensau...


In [30]:
# Texto de ingredientes que no se pudo leer → nulo
df.loc[df["n_ingredientes"] == 0, "n_ingredientes"] = np.nan

### 7.1 Resultados y decisiones

**Variables creadas:**
- 10 marcadores de ultraprocesado (`tiene_*`): aromas, edulcorantes, jarabes (glucosa, fructosa, maltodextrina, dextrosa), almidón modificado, grasa hidrogenada, proteína aislada, colorantes (E1xx), emulsionantes y espesantes (E4xx), potenciadores del sabor (E62x-63x) y aceite de palma.
- 4 marcadores de ingredientes añadidos y conservantes: `tiene_sal`, `tiene_azucar`, `tiene_aceite` y `tiene_conservantes` (E200-E252 y E280-E285; sin los acidulantes E260-E299).
- `n_ingredientes`: ingredientes principales contados desde el texto, sin los sub-ingredientes entre paréntesis. Los productos sin lista cuentan como 1.

**Correcciones** (revisando qué tags captura cada patrón, no solo cuántos productos):
- Colorantes: `en:e1\d\d` también detectaba los almidones modificados E14xx (`en:e1404` empieza por `en:e140`): 756 falsos positivos. Con `en:e1\d\d(?!\d)` solo detecta números de tres cifras.
- Grasa hidrogenada: `hydrogenated` también detectaba `non-hydrogenated` (aceite **no** hidrogenado): 44 de 274 productos. Se excluye con `(?<!non-)hydrogenated`.
- Conservantes: el rango E2xx incluye acidulantes (E260-E299: ácido acético, láctico, málico, CO₂). Nos quedamos con los conservantes, como se declaran en la etiqueta ("conservador: …").

**Separación por clase** (% de productos que tienen cada marcador):
- Los marcadores de ultraprocesado aparecen **casi exclusivamente en NOVA 4**: aromas en el 61 %, emulsionantes en el 47 %, jarabes en el 34 %. En las clases 1, 2 y 3 están prácticamente a 0.
- Los ingredientes añadidos **separan 1 de 3**: sal, azúcar o aceite aparecen en el 91 % de los NOVA 3 y solo en el 0,2 % de los NOVA 1.

| Marcador | NOVA 1 | NOVA 2 | NOVA 3 | NOVA 4 |
|---|---|---|---|---|
| Sal | 0,0 | 1,7 | 77,6 | 57,2 |
| Azúcar | 0,0 | 1,4 | 22,8 | 56,5 |
| Aceite | 0,2 | 6,7 | 39,2 | 39,3 |
| Conservantes | 0,6 | 0,6 | 9,3 | 22,9 |
| Sal, azúcar o aceite | 0,2 | 9,5 | 91,0 | 82,7 |

- La clase 2 sigue siendo la difícil para los marcadores: casi siempre es un solo ingrediente sin lista. Ahí trabajan `sin_lista_ingredientes`, la categoría y los nutrientes.
- `n_ingredientes` (media 1,8 / 1,1 / 5,2 / 11,3) y `additives_n` completan la separación.

**Limitación:** Open Food Facts calcula NOVA con reglas basadas en estos mismos ingredientes y aditivos. El modelo aprenderá en buena parte el algoritmo de OFF, no el criterio de una experta en nutrición.

**`n_ingredientes`:**
- 2 productos con 0 (texto ilegible) → nulo.
- 162 con más de 50 (multipacks, suplementos): se conservan; son NOVA 4 reales. Transformación logarítmica opcional en la fase 2.

**App:** las 14 variables se pueden identificar en una etiqueta española, porque los aditivos se declaran con su función (aroma, edulcorante, emulgente, colorante…). Serán casillas del formulario, que la lectura de la foto con Gemini pre-rellenará.

## Paso 8 · Guardar el dataset limpio

Guardamos un único CSV limpio y compartido en `data/processed/`. Cada una parte de él para su preproceso en la fase 2.

- Se conservan `code`, `product_name`, `brands` y `grupo_producto` (no entran al modelo: sirven para la app, el análisis de errores y la división por grupo).
- Se conserva `ingredients_text` para revisar errores del modelo.
- Se eliminan las listas de tags: ya están resumidas en las variables `tiene_*`.
- Se guarda comprimido (`.csv.gz`); pandas lo lee directamente.
- **Split fijo** (columna `split`): 80 % train / 20 % test con `StratifiedGroupKFold` (`random_state=42`). Estratificado por clase y agrupado por `grupo_producto`, para que un mismo producto no quede repartido entre train y test. **Todas usamos este split**: así los modelos se comparan con el mismo test.
- Agrupamos por `grupo_producto` (nombre + marca) y no por marca: agrupar por marca dejaría marcas enteras (miles de productos de marca blanca) solo en train o solo en test.

In [31]:
from sklearn.model_selection import StratifiedGroupKFold

# Listas de tags: ya convertidas en variables, no hacen falta en el CSV final
df_final = df.drop(columns=["ingredients_tags", "additives_tags", "ingredients_analysis_tags"]).reset_index(drop=True)

# Split fijo 80/20: estratificado por clase y sin repartir un mismo producto entre train y test
# random_state=42 → siempre sale la misma división
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df_final, df_final["nova_group"], groups=df_final["grupo_producto"]))

df_final["split"] = "train"
df_final.loc[test_idx, "split"] = "test"

RUTA = "../data/processed/off_nova_clean.csv.gz"
df_final.to_csv(RUTA, index=False)

import os
print(f"Guardado: {RUTA} ({os.path.getsize(RUTA) / 1e6:.1f} MB)")

Guardado: ../data/processed/off_nova_clean.csv.gz (6.7 MB)


In [32]:
# Verificación: lo releemos como lo hará cada una en la fase 2
check = pd.read_csv(RUTA, dtype={"code": str})

print(check.shape)
print("\nNulos que quedan (se imputan en la fase 2):")
print(check.isna().sum()[check.isna().sum() > 0])
print("\nClases:")
print(check["nova_group"].value_counts().sort_index())
check.dtypes

(48119, 35)

Nulos que quedan (se imputan en la fase 2):
product_name                        913
brands                             4107
ingredients_text                   4960
additives_n                           3
energy-kcal_100g                    590
fat_100g                            984
saturated-fat_100g                 1468
carbohydrates_100g                  958
sugars_100g                        1345
fiber_100g                        16088
proteins_100g                       966
salt_100g                          1321
fruits-vegetables-legumes_100g     5401
n_ingredientes                        2
dtype: int64

Clases:
nova_group
1     6190
2     5385
3     9831
4    26713
Name: count, dtype: int64


code                                  str
product_name                          str
brands                                str
ingredients_text                      str
additives_n                       float64
nova_group                          int64
pnns_groups_1                         str
pnns_groups_2                         str
energy-kcal_100g                  float64
fat_100g                          float64
saturated-fat_100g                float64
carbohydrates_100g                float64
sugars_100g                       float64
fiber_100g                        float64
proteins_100g                     float64
salt_100g                         float64
fruits-vegetables-legumes_100g    float64
sin_lista_ingredientes              int64
grupo_producto                        str
tiene_aromas                        int64
tiene_edulcorantes                  int64
tiene_jarabes                       int64
tiene_almidon_modificado            int64
tiene_grasa_hidrogenada           

In [33]:
# Comprobaciones del split
print(check["split"].value_counts(normalize=True).round(3))

# Proporción de clases en train y test: tienen que ser casi iguales
print(pd.crosstab(check["nova_group"], check["split"], normalize="columns").round(3))

# Ningún grupo de producto puede estar en los dos lados
grupos_train = set(check.loc[check["split"] == "train", "grupo_producto"])
grupos_test = set(check.loc[check["split"] == "test", "grupo_producto"])
print("Grupos en train y test a la vez:", len(grupos_train & grupos_test))   # tiene que ser 0

split
train    0.8
test     0.2
Name: proportion, dtype: float64
split        test  train
nova_group              
1           0.129  0.129
2           0.112  0.112
3           0.204  0.204
4           0.555  0.555
Grupos en train y test a la vez: 0
